In [1]:
%matplotlib inline
import random
import torch
from d2l import torch as d2l

In [2]:
class SyntheticRegressionData(d2l.DataModule):
    """Synthetic Data Module"""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32): # w = weights, b = bias (when all intercepts features are 0)
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val # total dataset
        self.X = torch.randn(n, len(w)) # n random samples with as many features as weights
        noise = torch.randn(n, 1) * noise # n random noise values scared by noise param
        self.y = torch.matmul(self.X, w.reshape((-1,1))) + b + noise #dotting X with the reshaped weight into a column vector

data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)

In [7]:
@d2l.add_to_class(SyntheticRegressionData) #want to iterate over data in min batches and use to update model
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train + self.num_val))
    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i: i + self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices] #use the indices to pull row from X and y then gives featuers and label tuple

In [8]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


In [10]:
#more efficient version (less python overhead)
@d2l.add_to_class(d2l.DataModule)  #@save
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size,
                                       shuffle=train)

@d2l.add_to_class(SyntheticRegressionData)  #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

len(data.train_dataloader()) # can also query length (number of batches)

32